In [4]:
# Load the same data
from langchain_community.document_loaders import DirectoryLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_PATH = "../data"

loader = DirectoryLoader(
    DATA_PATH,
    glob="**/*.csv",
    loader_cls=CSVLoader,
    loader_kwargs={"encoding": "utf-8"}
)

data = loader.load()

print(f"Loaded {len(data)} documents.")

Loaded 300 documents.


In [5]:
def run_chunking_experiment(data, chunk_size, chunk_overlap):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_documents(data)

    return chunks

In [6]:
experiments = [
    {"id": "C001", "chunk_size": 250, "chunk_overlap": 20},
    {"id": "C002", "chunk_size": 500, "chunk_overlap": 50},
    {"id": "C003", "chunk_size": 1000, "chunk_overlap": 100},
]

results = []

for experiment in experiments:

    chunks = run_chunking_experiment(
        data,
        experiment["chunk_size"],
        experiment["chunk_overlap"]
    )

    results.append({
        "Experiment": experiment["id"],
        "Chunk Size": experiment["chunk_size"],
        "Chunk Overlap": experiment["chunk_overlap"],
        "Documents": len(data),
        "Chunks": len(chunks)
    })

results

[{'Experiment': 'C001',
  'Chunk Size': 250,
  'Chunk Overlap': 20,
  'Documents': 300,
  'Chunks': 2983},
 {'Experiment': 'C002',
  'Chunk Size': 500,
  'Chunk Overlap': 50,
  'Documents': 300,
  'Chunks': 1538},
 {'Experiment': 'C003',
  'Chunk Size': 1000,
  'Chunk Overlap': 100,
  'Documents': 300,
  'Chunks': 900}]

In [8]:
import pandas as pd

experiment_df = pd.DataFrame(results)

experiment_df

,Experiment,Chunk Size,Chunk Overlap,Documents,Chunks
0,C001,250,20,300,2983
1,C002,500,50,300,1538
2,C003,1000,100,300,900


In [9]:
from pathlib import Path

experiment_log_path = Path("../experiments/experiment_log.csv")

experiment_df["Area"] = "Chunking"

experiment_df = experiment_df.rename(columns={
    "Experiment": "Experiment_ID",
    "Chunk Size": "Chunk_Size",
    "Chunk Overlap": "Chunk_Overlap"
})

experiment_df["Observation"] = [
    "Smallest chunk size; highest number of chunks",
    "Baseline configuration",
    "Largest chunk size; lowest number of chunks"
]

experiment_df["Conclusion"] = [
    "Smaller chunks produce more chunks",
    "Use as baseline for subsequent retrieval experiments",
    "Larger chunks produce fewer chunks"
]

experiment_df.to_csv(
    experiment_log_path,
    index=False
)

print(f"Experiment log saved to: {experiment_log_path}")

Experiment log saved to: ..\experiments\experiment_log.csv
